In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from core.signal.preprocess import *
from glob import glob
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os
import torchinfo
from core.nn.basic_dataloader import *

In [2]:
vctkds = torchaudio.datasets.VCTK_092(root="vctk/", download=False)
ds2 = VCTK_092("vctk/gen/csv")
class PairedDataset(torch.utils.data.Dataset):
    def __init__(self, ds1, ds2):
        self.ds1 = ds1
        self.ds2 = ds2
        assert len(ds1) == len(ds2), "Datasets must be the same length"
    def __len__(self):
        return len(self.ds1)
    def __getitem__(self, idx):
        wavA, srA, txtA, speakerA, dataA  = self.ds1[idx]
        wavB, srB, txtB, speakerB, dataB  = self.ds2[idx]
        # srA and srB should be the same, but just in case, return both
        # txt, speaker, and data should be the same, we don't really care about them (for now)
        newTuple = (wavA, wavB, srA, srB, txtA, speakerA, dataA)
        return newTuple
# paired dataset
paired_ds = PairedDataset(vctkds, ds2)
batch_size = 2
# Split into train and test
split = 0.85
lenTrain = int(len(paired_ds) * split)
lenTest = len(paired_ds) - lenTrain
# select lenTrain random indices
np.random.seed(0) # for reproducibility
idxtrain = np.random.choice(len(paired_ds), lenTrain, replace=False)
idxtest = np.setdiff1d(np.arange(len(paired_ds)), idxtrain)
train = torch.utils.data.Subset(paired_ds, idxtrain)
test = torch.utils.data.Subset(paired_ds, idxtest)
train_loader = torch.utils.data.DataLoader(train, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=batch_size, shuffle=False)

FileNotFoundError: [Errno 2] No such file or directory: 'vctk/gen/csv/txt'

In [ ]:
speaker_list = ['p225', 'p226','p227','p228', 'p229', 'p230', 'p231', 'p232', 'p233', 'p234', 'p236', 'p237', 'p238', 'p239', 'p240', 'p241', 'p243', 'p244', 'p245', 'p246', 'p247', 'p248', 'p249']
nSpeakers = len(speaker_list)
onehot_speaker = lambda x: torch.eye(nSpeakers)[speaker_list.index(x)]
# Is this a clean way to do this? Hell nah
# Is this efficient? Yes

In [ ]:
for (waveformA, waveformB, _, _, _, _, _) in train_loader:
    print(waveformB.shape)
    break

In [3]:
import torch
from torch import nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, init_features=32):
        super(UNet, self).__init__()
        
        features = init_features
        self.encoder1 = UNet._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool1d(kernel_size=4, stride=4)
        self.encoder2 = UNet._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.encoder3 = UNet._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.encoder4 = UNet._block(features * 4, features * 8, name="enc4")
        self.pool4 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.bottleneck = UNet._block(features * 8, features * 16, name="bottleneck")

        self.upconv4 = nn.ConvTranspose1d(features * 16, features * 8, kernel_size=2, stride=2)
        self.decoder4 = UNet._block((features * 8) * 2, features * 8, name="dec4")
        self.upconv3 = nn.ConvTranspose1d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = UNet._block((features * 4) * 2, features * 4, name="dec3")
        self.upconv2 = nn.ConvTranspose1d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = UNet._block((features * 2) * 2, features * 2, name="dec2")
        self.upconv1 = nn.ConvTranspose1d(features * 2, features, kernel_size=4, stride=4)
        self.decoder1 = UNet._block(features * 2, features, name="dec1")

        self.conv = nn.Conv1d(in_channels=features, out_channels=out_channels, kernel_size=1)

    @staticmethod
    def _block(in_channels, features, name):
        return nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(num_features=features),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_channels=features, out_channels=features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(num_features=features),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.upconv4(bottleneck)
        dec4 = self.center_crop(dec4, enc4.shape[2])
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = self.center_crop(dec3, enc3.shape[2])
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = self.center_crop(dec2, enc2.shape[2])
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = self.center_crop(dec1, enc1.shape[2])
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)

        return self.conv(dec1)

    def center_crop(self, dec, enc_shape):
        dec_shape = dec.shape[2]
        trim_size = (dec_shape - enc_shape) // 2
        # trim_size can be negative, in that case, we need to pad
        if trim_size < 0:
            pad_size = -trim_size
            dec = F.pad(dec, (pad_size, pad_size))
            return dec[:, :, pad_size:pad_size+enc_shape]        
        else:        
            return dec[:, :, trim_size:trim_size+enc_shape]

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet().to(device)


# Assuming the use of MSE loss and Adam optimizer
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

In [5]:

print_every = 100
num_epochs = 1  # for example, adjust as necessary
with tqdm(total=num_epochs * len(train_loader)) as pbar:
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, data in enumerate(train_loader, 0):
            # Unpack data
            inputs, labels = data[1], data[0]
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Print statistics
            running_loss += loss.item()
            if i % print_every == (print_every - 1):    # reset every 100 mini-batches
                running_loss = 0.0
            pbar.set_description(f'Epoch {epoch+1} Loss: {running_loss/(i % print_every + 1):.3f}')    
            pbar.update(1)

print('Finished Training')


NameError: name 'train_loader' is not defined

In [44]:
# Save model
torch.save(model.state_dict(), 'model_unet.pth')

In [51]:
# Test model
model.eval()
with torch.no_grad():
    for (waveformA, waveformB, _, _, _, _, _) in train_loader:
        outputs = model(waveformB.to(device))
        break

In [52]:
wav = outputs[1, :, :].cpu().detach()
torchaudio.save("test.wav", wav, 16000)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = model.to(device)
torchinfo.summary(net, (2, 2, 226209))

Layer (type:depth-idx)                   Output Shape              Param #
UNet                                     [2, 1, 226209]            --
├─Sequential: 1-1                        [2, 32, 226209]           --
│    └─Conv1d: 2-1                       [2, 32, 226209]           192
│    └─BatchNorm1d: 2-2                  [2, 32, 226209]           64
│    └─ReLU: 2-3                         [2, 32, 226209]           --
│    └─Conv1d: 2-4                       [2, 32, 226209]           3,072
│    └─BatchNorm1d: 2-5                  [2, 32, 226209]           64
│    └─ReLU: 2-6                         [2, 32, 226209]           --
├─MaxPool1d: 1-2                         [2, 32, 56552]            --
├─Sequential: 1-3                        [2, 64, 56552]            --
│    └─Conv1d: 2-7                       [2, 64, 56552]            6,144
│    └─BatchNorm1d: 2-8                  [2, 64, 56552]            128
│    └─ReLU: 2-9                         [2, 64, 56552]            --
│    └─

In [83]:
N_EPOCHS = 10
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, _, speaker_id, _) in train_loader:
            waveform = waveform.to(device)
            waveform = torch.cat([waveform, waveform], dim=1)
            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
            speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
            speaker_one_hot = speaker_one_hot.to(device)
            optimizer.zero_grad()
            output = net(waveform)
            loss = criterion(output, speaker_id)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")

  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.0232: 100%|██████████| 10/10 [12:46<00:00, 76.66s/it]


100%|██████████| 84/84 [00:06<00:00, 12.24it/s]

Accuracy: 93.89%


In [87]:
net_ds2 = ConvNet2D(nSpeakers).to(device)
N_EPOCHS = 10
optimizerds2 = torch.optim.Adam(net_ds2.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, _, speaker_id, _) in train_ds2_loader:
            waveform = waveform.to(device)
            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
            speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
            speaker_one_hot = speaker_one_hot.to(device)
            optimizerds2.zero_grad()
            output = net_ds2(waveform)
            loss = criterion(output, speaker_id)
            loss.backward()
            optimizerds2.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")

Loss: 0.2234: 100%|██████████| 10/10 [13:38<00:00, 81.88s/it]


In [94]:
# save the models

torch.save(net.state_dict(), "convnet2d.pt")
torch.save(net_ds2.state_dict(), "convnet2d_ds2.pt")

In [95]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_loader):
        waveform = waveform.to(device)
        waveform = torch.cat([waveform, waveform], dim=1)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_loader.dataset) * 100:.2f}%")

  0%|          | 0/84 [00:00<?, ?it/s]

100%|██████████| 84/84 [00:08<00:00,  9.84it/s]

Accuracy: 93.67%


In [96]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_ds2_loader):
        waveform = waveform.to(device)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net_ds2(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_ds2_loader.dataset) * 100:.2f}%")

100%|██████████| 84/84 [00:10<00:00,  8.26it/s]

Accuracy: 89.05%
